# Identifying Deepfakes - V25 Supervised PCA Forest

## The Supervised Rescue
This bypasses Isolation Forest and GMM entirely. We apply PCA to the highly volatile 512D neural space, squashing it into an ultra-dense 5D manifold. We then deploy a Supervised Random Forest Grid Search directly on that 5D space to physically map the boundaries.

In [ ]:
import os, warnings, zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, silhouette_score
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_PATH = '/content'; FOLDER_PATH = '/content/drive/MyDrive/DataMining/project_dataset'
else:
    BASE_PATH = './'; FOLDER_PATH = './project_dataset'

REAL_IMAGE_DIR, FAKE_IMAGE_DIR = os.path.join(BASE_PATH, 'Real-img'), os.path.join(BASE_PATH, 'Image')
RESOLUTION, BATCH_SIZE, SEED = 224, 64, 2026

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def extract_if_needed(zip_name, target_dir):
    if not os.path.exists(target_dir) and os.path.exists(os.path.join(FOLDER_PATH, zip_name)):
        with zipfile.ZipFile(os.path.join(FOLDER_PATH, zip_name), 'r') as z: z.extractall(BASE_PATH)

extract_if_needed('Real-img.zip', REAL_IMAGE_DIR)
extract_if_needed('Fake-img.zip', FAKE_IMAGE_DIR)

class DeepfakeDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.r = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.endswith(('.jpg', '.png'))] if os.path.exists(real_dir) else []
        self.f = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.endswith(('.jpg', '.png'))] if os.path.exists(fake_dir) else []
        self.all_files = self.r + self.f
        self.labels = [0]*len(self.r) + [1]*len(self.f)
        self.transform = transform
    def __len__(self): return len(self.all_files)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.all_files[idx]).convert('RGB')
            if self.transform: img = self.transform(img)
            return img, self.labels[idx], self.all_files[idx]
        except: return torch.zeros((3, RESOLUTION, RESOLUTION)), self.labels[idx], self.all_files[idx]

full_dataset = DeepfakeDataset(REAL_IMAGE_DIR, FAKE_IMAGE_DIR, transform=transforms.Compose([
    transforms.Resize((RESOLUTION, RESOLUTION)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
]))
all_idx = list(range(len(full_dataset)))
np.random.shuffle(all_idx)

NETWORK_POOL_SIZE = int(len(full_dataset) * 0.1)  
network_loader = DataLoader(Subset(full_dataset, all_idx[:NETWORK_POOL_SIZE]), batch_size=BATCH_SIZE, shuffle=True)
ml_loader = DataLoader(Subset(full_dataset, all_idx[NETWORK_POOL_SIZE:]), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for name, param in model.named_parameters():
    if any(layer_name in name for layer_name in ['conv1', 'bn1', 'layer1', 'layer2', 'layer3']): param.requires_grad = False
model.fc = nn.Linear(512, 2)
model = model.to(device)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
criterion = nn.CrossEntropyLoss()

if len(network_loader) > 0:
    for epoch in range(1):
        model.train()
        pbar = tqdm(network_loader, desc=f"Epoch 1/1")
        for imgs, labels, _ in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward(); optimizer.step()

In [ ]:
model.fc = nn.Identity()
model.eval()

latent_embeddings, ground_truth = [], []

with torch.no_grad():
    for imgs, labels, _ in tqdm(ml_loader, desc="Extraction"):
        imgs = imgs.to(device)
        latent = model(imgs)
        latent_embeddings.append(latent.cpu().numpy())
        ground_truth.extend(labels.numpy())

if len(latent_embeddings) > 0:
    latent_embeddings = np.vstack(latent_embeddings)
    ground_truth = np.array(ground_truth)

In [ ]:
print("Phase 1 Finetuning: K-Means Silhouette Grid Search...")
k_candidates = [2, 3, 5, 10, 15]
best_k, best_score = 5, -1

subset_size = min(5000, len(latent_embeddings)) 
subset_x = latent_embeddings[:subset_size]

for k in k_candidates:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    preds = km.fit_predict(subset_x)
    if len(set(preds)) > 1:
        score = silhouette_score(subset_x, preds)
        print(f"  k={k} | Silhouette Score: {score:.4f}")
        if score > best_score: best_score, best_k = score, k

print(f"--> Opting for Optimal Clusters: {best_k}\\n")
cluster_labels = KMeans(n_clusters=best_k, random_state=SEED, n_init=10).fit_predict(latent_embeddings)

In [ ]:
try:
    global_y_true, global_y_pred, global_y_prob = [], [], []
    print("Executing Dense Supervised Random Forest...\\n")
    
    for i in range(best_k):
        mask = (cluster_labels == i)
        X_cluster = latent_embeddings[mask]
        y_cluster = ground_truth[mask]
        
        if len(np.unique(y_cluster)) < 2: continue
            
        print(f"--- Macro-Cluster {i} (N={len(y_cluster)}, Fakes={np.sum(y_cluster)}) ---")

        # Densify Dimensions
        pca = PCA(n_components=min(5, X_cluster.shape[1]), random_state=SEED)
        X_pca = pca.fit_transform(X_cluster)
        
        X_train, X_test, y_train, y_test = train_test_split(X_pca, y_cluster, test_size=0.2, random_state=SEED)
        
        rf = RandomForestClassifier(random_state=SEED)
        param_grid = {'n_estimators': [50, 100], 'max_depth': [5, 10, None]}
        clf = GridSearchCV(rf, param_grid, cv=3)
        clf.fit(X_train, y_train)
        
        print(f"  Optimal RF Config: {clf.best_params_}")
        final_preds = clf.predict(X_test)
        final_probs = clf.predict_proba(X_test)[:, 1]
        
        global_y_true.extend(y_test)
        global_y_pred.extend(final_preds)
        global_y_prob.extend(final_probs)
        
        print(f"  Local RF Accuracy: {accuracy_score(y_test, final_preds):.3f} | Local AUC: {roc_auc_score(y_test, final_probs):.3f}\\n")
        
except Exception as e: print(f"Error: {e}")

In [ ]:
try:
    g_acc = accuracy_score(global_y_true, global_y_pred)
    g_prec = precision_score(global_y_true, global_y_pred)
    g_rec = recall_score(global_y_true, global_y_pred)
    g_f1 = f1_score(global_y_true, global_y_pred)
    g_auc = roc_auc_score(global_y_true, global_y_prob)
    print("="*40)
    print(f"FINAL SUPERVISED HYBRID METRICS (PCA + RANDOM FOREST)")
    print("="*40)
    print(f"Global Accuracy:   {g_acc:.4f}")
    print(f"Global Precision:  {g_prec:.4f}")
    print(f"Global Recall:     {g_rec:.4f}")
    print(f"Overall F1 Score:  {g_f1:.4f}")
    print(f"Cumulative AUC:    {g_auc:.4f}")
    print("="*40)
except: pass